In [ ]:
import os, json, shutil, subprocess, torch
from google.colab import drive

assert torch.cuda.is_available(), "Chua bat GPU T4"
print("GPU:", torch.cuda.get_device_name(0))
drive.mount("/content/drive")

REPO_DRIVE = "/content/drive/MyDrive/NghiepVuTM/CT255_RumorDetection_BiGCN_Plus"
REPO_LOCAL = "/content/CT255_RumorDetection_BiGCN_Plus"
DATA_LOCAL = "/content/data_local/rumor_detection_acl2017"
DRIVE_OUT = f"{REPO_DRIVE}/results/multi_seed"

for sub in ("src", "scripts", "configs"):
    shutil.copytree(f"{REPO_DRIVE}/{sub}", f"{REPO_LOCAL}/{sub}", dirs_exist_ok=True)

if not os.path.exists(f"{DATA_LOCAL}/twitter15"):
    raw_candidates = [
        "/content/drive/MyDrive/NghiepVuTM/dataset/rumdetect2017/rumor_detection_acl2017",
        "/content/drive/MyDrive/NghiepVuTM/dataset/rumor_detection_acl2017",
    ]
    raw_src = next((p for p in raw_candidates if os.path.exists(f"{p}/twitter15")), None)
    assert raw_src is not None, "Khong tim thay rumdetect2017 tren Drive"
    os.makedirs(os.path.dirname(DATA_LOCAL), exist_ok=True)
    shutil.copytree(raw_src, DATA_LOCAL, dirs_exist_ok=True)

os.makedirs(DRIVE_OUT, exist_ok=True)
os.chdir(REPO_LOCAL)
subprocess.run(["pip", "install", "-q", "scikit-learn", "pyyaml", "scipy", "pandas", "matplotlib"], check=True)
print("Ready:", os.listdir(DATA_LOCAL))

In [ ]:
SEEDS = [42, 43, 44, 45, 46]
DATASETS = ["Twitter16", "Twitter15"]

def run_models(models, datasets=DATASETS, seeds=SEEDS, epochs=100, snapshot_k=5):
    expected = len(seeds) * 5
    for ds in datasets:
        for m in models:
            tag = f"{ds}_{m}"
            d_dir = os.path.join(DRIVE_OUT, tag)
            l_dir = os.path.join("results", "multi_seed", tag)
            j_path = os.path.join(d_dir, "results_multiseed.json")
            if os.path.exists(j_path):
                with open(j_path, "r", encoding="utf-8") as f:
                    info = json.load(f)
                if (
                    info.get("summary", {}).get("n_runs") == expected
                    and info.get("seeds") == list(seeds)
                    and "cuda" in str(info.get("device", ""))
                ):
                    print(f"[SKIP] {tag}")
                    shutil.copytree(d_dir, l_dir, dirs_exist_ok=True)
                    continue
            print(f"[RUN] {tag}")
            subprocess.run(
                [
                    "python", "scripts/run_benchmark.py",
                    "--data_root", DATA_LOCAL,
                    "--dataset", ds,
                    "--models", m,
                    "--seeds", *[str(s) for s in seeds],
                    "--epochs", str(epochs),
                    "--snapshot_k", str(snapshot_k),
                    "--device", "cuda",
                    "--save_dir", "results/multi_seed",
                ],
                check=True,
            )
            shutil.copytree(l_dir, d_dir, dirs_exist_ok=True)

In [ ]:
run_models(["TF-IDF-Ridge", "TF-IDF-LR", "TF-IDF-SVC", "MLP-BoW", "GCN-Single", "GraphSAGE-Single", "GAT-Single"])

In [ ]:
run_models(["BiGCN-Base", "BiGCN-NoRoot", "BiGCN-NoDropEdge", "BiGCN-TDOnly", "BiGCN-BUOnly"])

In [ ]:
run_models(["BiGCN-Plus-UserID", "BiGCN-Plus-Clean", "BiGCN-Plus-B1Only", "BiGCN-Plus-B2Only", "BiGCN-Plus-B12"])

In [ ]:
run_models(["BiGCN-Base+", "EBGCN", "RAGCL"])

In [ ]:
shutil.copytree(DRIVE_OUT, "results/multi_seed", dirs_exist_ok=True)
subprocess.run(["python", "scripts/aggregate_results.py", "--results_dir", "results/multi_seed", "--out_dir", "results"], check=True)
shutil.copytree("results/tables", f"{REPO_DRIVE}/results/tables", dirs_exist_ok=True)
shutil.copytree("results/figures", f"{REPO_DRIVE}/results/figures", dirs_exist_ok=True)
for fn in ("master_multiseed_summary.md", "same_eval_comparison.md", "statistical_tests.md"):
    print(f"\n===== {fn} =====")
    with open(f"results/tables/{fn}", "r", encoding="utf-8") as f:
        print(f.read())